In [1]:
import uproot
import numpy as np
import pandas as pd
import awkward as ak
import matplotlib.pyplot as plt

ROOT_DATA_DIR = "../mu3e_root_data"
signal_data_file = f"{ROOT_DATA_DIR}/run42_sig.root"
background_data_file = f"{ROOT_DATA_DIR}/run42_bg.root"

HIT_COUNT_CUTOFF = 128


with uproot.open(background_data_file) as file:
    bg_sensor_positions = file["alignment/sensors"].arrays(library="pd")
    bg_mppc_positions = file["alignment/mppcs"].arrays(library="pd")
    bg_fibre_positions = file["alignment/fibres"].arrays(library="pd")
    bg_event_data = file["mu3e"].arrays(library="pd")
with uproot.open(signal_data_file) as file:
    sig_sensor_positions = file["alignment/sensors"].arrays(library="pd")
    sig_mppc_positions = file["alignment/mppcs"].arrays(library="pd")
    sig_fibre_positions = file["alignment/fibres"].arrays(library="pd")
    sig_event_data = file["mu3e"].arrays(library="pd")

In [2]:
def reorder_nla(nla: np.ndarray, padding_value: int = -1) -> np.ndarray:
    """
    Reorders the NLA array to ensure that non-padded entries are at the beginning.
    Assumes padding is identifiable via `nla[:, :, 0] == padding_value`.
    """
    # Identify valid entries
    if nla.ndim == 3:
        valid_mask = nla[:, :, 0] != padding_value
        B, N, D = nla.shape
        flat_nla = nla.reshape(B * N, D)
        flat_valid_mask = valid_mask.reshape(B * N)

        # Get indices of valid entries
        valid_indices = np.nonzero(flat_valid_mask)[0]

        # Allocate output
        reordered_nla = np.full_like(nla, padding_value)

        counts = valid_mask.sum(axis=1)


        # Fill output using advanced indexing
        row_ids = np.repeat(np.arange(B), counts)
        group_counts = np.bincount(row_ids, minlength=B)

        # Compute start indices for placing data
        start_idx = np.zeros_like(group_counts)
        np.cumsum(group_counts[:-1], out=start_idx[1:])

        # Where to write valid entries in each row
        insert_pos = np.hstack([np.arange(c) for c in group_counts])
        reordered_nla[row_ids, insert_pos] = flat_nla[valid_indices]

    else:
        valid_mask = nla != padding_value
        B, N = nla.shape
        flat_nla = nla.reshape(B * N)
        flat_valid_mask = valid_mask.reshape(B * N)

        # Get indices of valid entries
        valid_indices = np.nonzero(flat_valid_mask)[0]

        # Allocate output
        reordered_nla = np.full_like(nla, padding_value)

        counts = valid_mask.sum(axis=1)
        # Fill output using advanced indexing
        row_ids = np.repeat(np.arange(B), counts)
        group_counts = np.bincount(row_ids, minlength=B)

        # Compute start indices for placing data
        start_idx = np.zeros_like(group_counts)
        np.cumsum(group_counts[:-1], out=start_idx[1:])

        # Where to write valid entries in each row
        insert_pos = np.hstack([np.arange(c) for c in group_counts])
        reordered_nla[row_ids, insert_pos] = flat_nla[valid_indices]


    # Compute number of valid entries per batch
    counts = valid_mask.sum(axis=1)
    # Flatten for easier fancy indexing

    return reordered_nla



In [3]:
def load_ak_series_to_numpy(
    series: pd.Series, max_cols: int = 256, fill_value: int = -1
) -> np.ndarray:
    """
    Converts an Awkward Array Series to a padded NumPy array (2D).
    Pads each row to `max_cols`, truncates longer rows, and ignores empty arrays.
    """
    # Combine series into one awkward array
    ak_array = ak.Array(series.to_list())  # safe if series contains ak Arrays or lists

    # Filter out arrays of invalid lengths
    lengths = ak.num(ak_array)
    mask = (lengths > 0) & (lengths <= max_cols)
    ak_array = ak_array[mask]

    # Clip longer arrays (optional depending on your needs)
    ak_array = ak.pad_none(ak_array, max_cols, clip=True)

    # Replace None with fill_value
    ak_array_filled = ak.fill_none(ak_array, fill_value)

    # Convert to NumPy
    result = ak.to_numpy(ak_array_filled)
    return result

In [4]:
def load_event_ak_to_numpy(
    pixel_series: pd.Series,
    mppc_series: pd.Series,
    pixel_hit_cutoff: int = 256,
    mppc_hit_cutoff: int = 256,
    fill_value: int = -1,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Converts pixel and MPPC data from Awkward Array Series to padded NumPy arrays.
    Pads each row to `pixel_hit_cutoff` and `mppc_hit_cutoff`, truncates longer rows,
    and ignores empty arrays.
    """
    pixel_ak_array = ak.Array(pixel_series.to_list())
    mppc_ak_array = ak.Array(mppc_series.to_list())

    # Filter out arrays of invalid lengths
    pixel_lengths = ak.num(pixel_ak_array)
    mppc_lengths = ak.num(mppc_ak_array)
    pixel_mask = (pixel_lengths > 0) & (pixel_lengths <= pixel_hit_cutoff)
    mppc_mask = (mppc_lengths > 0) & (mppc_lengths <= mppc_hit_cutoff)
    mask = pixel_mask & mppc_mask
    pixel_ak_array = pixel_ak_array[mask]
    mppc_ak_array = mppc_ak_array[mask]
    # Clip longer arrays
    pixel_ak_array = ak.pad_none(pixel_ak_array, pixel_hit_cutoff, clip=True)
    mppc_ak_array = ak.pad_none(mppc_ak_array, mppc_hit_cutoff, clip=True)
    # Replace None with fill_value
    pixel_ak_array_filled = ak.fill_none(pixel_ak_array, fill_value)
    mppc_ak_array_filled = ak.fill_none(mppc_ak_array, fill_value)

    # Convert to NumPy
    pixel_data = ak.to_numpy(pixel_ak_array_filled)
    mppc_data = ak.to_numpy(mppc_ak_array_filled)
    
    return pixel_data, mppc_data
    

In [5]:
def convert_mppc_to_location(
    mppc: np.ndarray, mppc_positions: pd.DataFrame, padding_value: float = -1
) -> np.ndarray:
    """
    Converts a 1D array of MPPC IDs to their corresponding positions in space.
    The MPPC IDs are expected to be in the 'mppc' column of the mppc_positions DataFrame.
    The function returns a 2D array of positions with shape (N, 3), where N is the number of MPPCs.
    Each row corresponds to the (cx, cy, cz) coordinates of an MPPC.

    Parameters:
    mppc (np.ndarray): 1D array of MPPC IDs.
    mppc_positions (pd.DataFrame): DataFrame containing MPPC positions with columns ['mppc', 'cx', 'cy', 'cz'].

    Returns:
    np.ndarray: 2D array of shape (N, 3) containing the (vx, vy) coordinates of the MPPCs.
    """
    # Validate input
    if not isinstance(mppc, np.ndarray) or mppc.ndim != 2:
        raise ValueError("mppc must be a 2D numpy array.")

    required_columns = ["mppc", "vx", "vy"]

    if not all(col in mppc_positions.columns for col in required_columns):
        raise ValueError(
            f"mppc_positions DataFrame must contain the following columns: {required_columns}"
        )

    mppc_positions = mppc_positions.set_index("mppc")
    locations = np.full((*mppc.shape, 2), dtype=float, fill_value=padding_value)
    flat_locations = locations.reshape(-1, 2)

    flatted_mppc = mppc.flatten()
    flat_mask = flatted_mppc != padding_value

    mppc_data = mppc_positions.loc[flatted_mppc[flat_mask]]

    vx = mppc_data["vx"].to_numpy()
    vy = mppc_data["vy"].to_numpy()

    flat_locations[flat_mask] = np.column_stack((vx, vy))

    return locations

In [6]:
def convert_pid_to_location(
    pixel_id: np.ndarray,
    sensor_positions: pd.DataFrame,
    padding_value: float = -1,
    sensor_fault_rate=0.0,
) -> np.ndarray:
    if sensor_positions.empty:
        raise ValueError("sensor_positions DataFrame is empty.")

    required_columns = [
        "id",
        "vx",
        "vy",
        "vz",
        "rowx",
        "rowy",
        "rowz",
        "colx",
        "coly",
        "colz",
    ]
    if not all(col in sensor_positions.columns for col in required_columns):
        raise ValueError(f"Missing required columns: {required_columns}")

    # Preprocessing
    sensor_positions = sensor_positions.set_index("id", drop=False)
    valid_mask = pixel_id != padding_value

    # Decode pixel_id to chip, col, row
    hit_chip_id = (pixel_id // 2**16).astype(np.int32)
    hit_col_id = ((pixel_id // 2**8) % 2**8).astype(np.float32)
    hit_row_id = (pixel_id % 2**8).astype(np.float32)

    # Flatten inputs for easier indexing
    flat_valid_mask = valid_mask.flatten()
    flat_chip_id = hit_chip_id.flatten()
    flat_col_id = hit_col_id.flatten()
    flat_row_id = hit_row_id.flatten()

    keep_mask = flat_chip_id // 2**12 == 0

    layer_id = ((flat_chip_id // 2**10) % 4) + 1

    # Apply sensor fault rate
    if sensor_fault_rate > 0:
        # If sensor_fault_rate is applied, we keep the mask for faulty sensors
        # but also ensure that layer 1 and 2 are always kept
        # This is because we only assume layers 3 and 4 to be faulty
        keep_mask = (
            (np.random.rand(len(sensor_data)) >= sensor_fault_rate)
            | (layer_id == 1 | layer_id == 2)
        ) & keep_mask

    flat_valid_mask = flat_valid_mask & keep_mask

    # Filter only valid pixel ids
    valid_chip_ids = flat_chip_id[flat_valid_mask]
    valid_cols = flat_col_id[flat_valid_mask] + 0.5
    valid_rows = flat_row_id[flat_valid_mask] + 0.5

    # Lookup transformation vectors
    try:
        sensor_data = sensor_positions.loc[valid_chip_ids]
    except KeyError:
        raise ValueError("Some chip IDs not found in sensor_positions.")

    # Compute positions
    vx = sensor_data["vx"].values
    vy = sensor_data["vy"].values
    vz = sensor_data["vz"].values
    rowx = sensor_data["rowx"].values
    rowy = sensor_data["rowy"].values
    rowz = sensor_data["rowz"].values
    colx = sensor_data["colx"].values
    coly = sensor_data["coly"].values
    colz = sensor_data["colz"].values

    x = vx + valid_cols * colx + valid_rows * rowx
    y = vy + valid_cols * coly + valid_rows * rowy
    z = vz + valid_cols * colz + valid_rows * rowz

    # Fill output array
    location = np.full((*pixel_id.shape, 3), padding_value, dtype=np.float64)
    flat_location = location.reshape(-1, 3)
    flat_location[flat_valid_mask] = np.stack([x, y, z], axis=1)

    return location


def compute_distance(data: np.ndarray, padding_value=-1) -> np.ndarray:
    distance_data = np.linalg.norm(data, axis=-1)
    distance_data[(data == padding_value).all(axis=-1)] = -1
    return distance_data


def sort_by_feature(
    data: np.ndarray, feature: np.ndarray, padding_value=-1
) -> np.ndarray:
    """
    Vectorized version: Sorts `data` along axis 1 by `feature`, respecting a padding mask.
    """
    if data.shape[0:2] != feature.shape[0:2]:
        raise ValueError(
            "Data and feature must have the same number of rows and columns."
        )

    # Create a valid mask over first 2 dims
    if data.ndim == 3:
        valid_mask = (data != padding_value).any(axis=-1)  # shape (B, N)
    elif data.ndim == 2:
        valid_mask = data != padding_value
    else:
        raise ValueError("Data must be 2D or 3D array.")

    # Number of valid elements per row
    num_valid = valid_mask.sum(axis=1)

    # argsort feature, ignoring invalid entries
    sort_indices = np.argsort(np.where(valid_mask, feature, np.inf), axis=1)

    # Prepare output filled with padding_value
    sorted_data = np.full_like(data, padding_value)

    # Use np.take_along_axis to reorder data
    if data.ndim == 3:
        sorted_full = np.take_along_axis(data, sort_indices[:, :, None], axis=1)
    elif data.ndim == 2:
        sorted_full = np.take_along_axis(data, sort_indices[:, :], axis=1)
    else:
        raise ValueError("Data must be 2D or 3D array.")

    # Mask out only the valid sorted parts
    for i, n_valid in enumerate(num_valid):
        if n_valid > 0:
            sorted_data[i, :n_valid] = sorted_full[i, :n_valid]

    return sorted_data

In [7]:
def adjust_timestamps(timestamps: np.ndarray, padding_value: int = -1) -> np.ndarray:
    """
    Adjusts timestamps based on the data mask.
    If data is padded, the corresponding timestamps are set to -1.
    """
    masked_timestamps = np.where(timestamps != padding_value, timestamps, np.inf)
    min_timestamps = masked_timestamps.min(axis=1, keepdims=True)
    adjusted_timestamps = np.where(
        timestamps != padding_value, timestamps - min_timestamps, -1
    )
    return adjusted_timestamps

In [8]:

# Load background data
pixel_bg_data, mppc_bg_data = load_event_ak_to_numpy(
    bg_event_data["hit_pixelid"],
    bg_event_data["fibremppc_mppc"],
    pixel_hit_cutoff=HIT_COUNT_CUTOFF,
    mppc_hit_cutoff=HIT_COUNT_CUTOFF,
    fill_value=-1,
)
pixel_sig_data, mppc_sig_data = load_event_ak_to_numpy(
    sig_event_data["hit_pixelid"],
    sig_event_data["fibremppc_mppc"],
    pixel_hit_cutoff=HIT_COUNT_CUTOFF,
    mppc_hit_cutoff=HIT_COUNT_CUTOFF,
    fill_value=-1,
)
pixel_bg_timestamps, mppc_bg_timestamps = load_event_ak_to_numpy(
    bg_event_data["hit_timestamp"],
    bg_event_data["fibremppc_timestamp"],
    pixel_hit_cutoff=HIT_COUNT_CUTOFF,
    mppc_hit_cutoff=HIT_COUNT_CUTOFF,
    fill_value=-1,
)
pixel_sig_timestamps, mppc_sig_timestamps = load_event_ak_to_numpy(
    sig_event_data["hit_timestamp"],
    sig_event_data["fibremppc_timestamp"],
    pixel_hit_cutoff=HIT_COUNT_CUTOFF,
    mppc_hit_cutoff=HIT_COUNT_CUTOFF,
    fill_value=-1,
)


In [9]:
pixel_bg_data_positions = convert_pid_to_location(
    pixel_bg_data, bg_sensor_positions, padding_value=-1
)
pixel_sig_data_positions = convert_pid_to_location(
    pixel_sig_data, sig_sensor_positions, padding_value=-1
)
pixel_bg_data_spacetime = np.concatenate(
    (pixel_bg_data_positions, adjust_timestamps(pixel_bg_timestamps)[..., np.newaxis]), axis=-1
)
pixel_sig_data_spacetime = np.concatenate(
    (pixel_sig_data_positions, adjust_timestamps(pixel_sig_timestamps)[..., np.newaxis]), axis=-1
)

In [10]:
mppc_bg_data_positions = convert_mppc_to_location(
    mppc_bg_data, bg_mppc_positions, padding_value=-1
)
mppc_sig_data_positions = convert_mppc_to_location(
    mppc_sig_data, sig_mppc_positions, padding_value=-1
)
mppc_bg_data_spacetime = np.concatenate(
    (mppc_bg_data_positions, adjust_timestamps(mppc_bg_timestamps)[..., np.newaxis]), axis=-1
)
mppc_sig_data_spacetime = np.concatenate(
    (mppc_sig_data_positions, adjust_timestamps(mppc_sig_timestamps)[..., np.newaxis]), axis=-1
)

In [11]:
pixel_bg_data_spacetime[(pixel_bg_data_positions == -1).all(axis=-1), -1] = -1
pixel_sig_data_spacetime[(pixel_sig_data_positions == -1).all(axis=-1), -1] = -1
pixel_bg_data_spacetime = reorder_nla(pixel_bg_data_spacetime, padding_value=-1)
pixel_sig_data_spacetime = reorder_nla(pixel_sig_data_spacetime, padding_value=-1)

In [12]:
mppc_bg_data_spacetime[(mppc_bg_data_positions == -1).all(axis=-1), -1] = -1
mppc_sig_data_spacetime[(mppc_sig_data_positions == -1).all(axis=-1), -1] = -1
mppc_bg_data_spacetime = reorder_nla(mppc_bg_data_spacetime, padding_value=-1)
mppc_sig_data_spacetime = reorder_nla(mppc_sig_data_spacetime, padding_value=-1)


In [13]:
pixel_bg_data_spacetime = sort_by_feature(
    pixel_bg_data_spacetime, pixel_bg_data_spacetime[:, :, -1], padding_value=-1
)
pixel_sig_data_spacetime = sort_by_feature(
    pixel_sig_data_spacetime, pixel_sig_data_spacetime[:, :, -1], padding_value=-1
)
mppc_bg_data_spacetime = sort_by_feature(
    mppc_bg_data_spacetime, mppc_bg_data_spacetime[:, :, -1], padding_value=-1
)
mppc_sig_data_spacetime = sort_by_feature(
    mppc_sig_data_spacetime, mppc_sig_data_spacetime[:, :, -1], padding_value=-1
)

In [14]:
DATA_DIR = "../mu3e_trigger_data"

np.save(f"{DATA_DIR}/run42_bg_pixel_128.npy", pixel_bg_data_spacetime)
np.save(f"{DATA_DIR}/run42_sig_pixel_128.npy", pixel_sig_data_spacetime)
np.save(f"{DATA_DIR}/run42_bg_mppc_128.npy", mppc_bg_data_spacetime)
np.save(f"{DATA_DIR}/run42_sig_mppc_128.npy", mppc_sig_data_spacetime)

In [15]:
def convert_pixel_id_to_nla(
    pixel_id: np.ndarray, padding_value: int = -1
) -> np.ndarray:
    nla = np.full((*pixel_id.shape, 4), padding_value, dtype=np.int32)
    valid_mask = pixel_id != padding_value

    chip_id = pixel_id // 2**16
    station = chip_id // 2**12
    layer = ((chip_id // 2**10) % 4) + 1
    phi = ((chip_id // 2**5) % 2**5) + 1
    z_prime = chip_id % 2**5

    z = np.where(layer == 3, z_prime - 7, np.where(layer == 4, z_prime - 6, z_prime))

    station_mask = station == 0
    valid_mask = valid_mask & station_mask

    nla[valid_mask, 0] = station[valid_mask]
    nla[valid_mask, 1] = layer[valid_mask]
    nla[valid_mask, 2] = phi[valid_mask]
    nla[valid_mask, 3] = z[valid_mask]

    return nla